# MVP — Pipeline de Dados em Nuvem

Este notebook demonstra um fluxo ETL: extração, transformação, validação e carregamento. Nesta versão, os arquivos ficam no ambiente do Colab; a integração com armazenamento de nuvem é uma evolução futura.


In [ ]:
import json
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/content/pipeline-dados-nuvem')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs'
for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Pastas do projeto preparadas em:', BASE_DIR)


## 1. Criar dados de exemplo

Se você tiver um CSV próprio, envie-o para `data/raw/vendas.csv`, usando as colunas esperadas descritas no README. Para facilitar o primeiro teste, esta célula cria dados fictícios com duplicata e registros inválidos.


In [ ]:
sample_data = [
    {'id_venda': 1, 'data': '2026-09-01', 'produto': 'Caderno', 'quantidade': 2, 'valor_unitario': 15.5},
    {'id_venda': 2, 'data': '2026-09-02', 'produto': 'Caneta', 'quantidade': 5, 'valor_unitario': 3.0},
    {'id_venda': 2, 'data': '2026-09-02', 'produto': 'Caneta', 'quantidade': 5, 'valor_unitario': 3.0},
    {'id_venda': 3, 'data': '2026-09-03', 'produto': 'Mochila', 'quantidade': 1, 'valor_unitario': 120.0},
    {'id_venda': 4, 'data': '', 'produto': 'Lápis', 'quantidade': 10, 'valor_unitario': 1.5},
    {'id_venda': 5, 'data': '2026-09-05', 'produto': 'Borracha', 'quantidade': -2, 'valor_unitario': 2.0},
]
input_file = RAW_DIR / 'vendas.csv'
if not input_file.exists():
    pd.DataFrame(sample_data).to_csv(input_file, index=False)

df = pd.read_csv(input_file)
print('Prévia dos dados de entrada:')
display(df.head(10))
print('Total de registros:', len(df))


## 2. Transformar e validar

O pipeline padroniza os nomes das colunas, remove duplicatas, converte os tipos de dados e descarta registros que não atendem às regras básicas de qualidade.


In [ ]:
rows_input = len(df)
df.columns = df.columns.astype(str).str.strip().str.lower().str.replace(' ', '_', regex=False)
required = ['id_venda', 'data', 'produto', 'quantidade', 'valor_unitario']
missing_columns = [col for col in required if col not in df.columns]
if missing_columns:
    raise ValueError(f'Colunas obrigatórias ausentes: {missing_columns}')

duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df['id_venda'] = pd.to_numeric(df['id_venda'], errors='coerce')
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['produto'] = df['produto'].astype('string').str.strip()
df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')
df['valor_unitario'] = pd.to_numeric(df['valor_unitario'], errors='coerce')

valid_mask = (
    df['id_venda'].notna() & df['data'].notna() & df['produto'].notna()
    & df['produto'].ne('') & df['quantidade'].notna()
    & df['valor_unitario'].notna() & df['quantidade'].ge(0)
    & df['valor_unitario'].ge(0)
)
invalid_rows = int((~valid_mask).sum())
clean = df.loc[valid_mask].copy()
clean['id_venda'] = clean['id_venda'].astype('int64')
clean['quantidade'] = clean['quantidade'].astype('int64')
clean['valor_total'] = (clean['quantidade'] * clean['valor_unitario']).round(2)
clean['data'] = clean['data'].dt.strftime('%Y-%m-%d')

print('Prévia dos dados tratados:')
display(clean)


## 3. Carregar os resultados

Os dados processados são salvos em CSV, e as métricas da execução são registradas em JSON.


In [ ]:
output_file = PROCESSED_DIR / 'vendas_tratadas.csv'
report_file = OUTPUT_DIR / 'relatorio_qualidade.json'
clean.to_csv(output_file, index=False)

report = {
    'status': 'sucesso',
    'registros_entrada': int(rows_input),
    'duplicatas_removidas': duplicates_removed,
    'registros_invalidos_descartados': invalid_rows,
    'registros_saida': int(len(clean)),
    'percentual_retido': round(len(clean) / rows_input * 100, 2) if rows_input else 0.0,
    'arquivo_saida': str(output_file),
    'colunas_saida': list(clean.columns),
}
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print('\nArquivos gerados:')
print(output_file)
print(report_file)


## 4. Baixar os resultados (opcional)

Execute esta célula no Google Colab se quiser baixar o CSV tratado e o relatório JSON para o seu computador.


In [ ]:
from google.colab import files
files.download(str(output_file))
files.download(str(report_file))
